# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [22]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}

# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

1. The lectures cover the command line and Python basics (01)
2.  Git, functions, and files (02)
3.  environments, shell pipelines, and NumPy (03)
4.  Jupyter and labeled pandas data (04)
5. data contracts, cleaning, and validation (05)
6. joins, reshaping, and concatenation (06)

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [6]:
%%bash
# Count number of lines in every `.csv` file inside `data/` and save results in `file_inventory`
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [7]:
# Write a function, such as profile(name, table), that returns the four values in README
def profile(name, table):
    """returns the four values in README"""
    dict = {'table': name,
              'rows': table.shape[0],
              'columns': table.shape[1],
              'exact_duplicate_rows': table.duplicated().sum()}
    return dict

# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.
    # a) Create an empty list called `results`
results = []

    # b) For each file in `files`
for name in manifest["files"]:
        # (i) Apply the function `profile`
    data = profile(name, raw[Path(name).stem])
        # (ii) And append the results to the list `results`
    results.append(data)

    # c) Convert list of profiles into a dataframe
results_df = pd.DataFrame(results)

    # d) And save this as a csv in PROFILE_PATH without index indicators. 
results_df.to_csv(PROFILE_PATH, index = False)


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**TODO:** Replace this text with the contract: the intended grain (rows represent) and key (single or composite; which column uniquely identified the row) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

### 2.2 Stack the two visit files

In [144]:
# Rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
    # a) Rename the 'wt_kg' column to 'weight_kg'
        # (i) Create a copy for 'visits_south'
visits_south = raw["visits_south"].copy(deep = True)
        # (ii) Rename the 'wt_kg' column in the copy of `visits_south`
visits_south.rename(columns = {'wt_kg': "weight_kg"}, inplace=True)
        # (iii) Add `source_file` to table
visits_south['source_file'] = 'south'  

     # b) Rename the 'wt_kg' column to 'weight_kg'
            # (i) Create a copy for 'visits_south'
visits_north = raw["visits_north"].copy(deep = True)
            # (ii) Rename the 'wt_kg' column in the copy of `visits_south`
visits_north.rename(columns = {'wt_kg': "weight_kg"}, inplace=True)
            # (iii) Add `source_file` to table
visits_north['source_file'] = 'north'  
    # c) Concatenate the north and south tables, north table first
visits_raw = pd.concat([visits_north, visits_south], ignore_index = True)

    # d) Print concatenated raw visits table
visits_raw

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,north
1,P00002,week12,2026-05-08,15:20,129,82.0,north
2,P00003,baseline,2026-01-16,11:45,148,67.5,north
3,P00003,wk12,2026-04-07,10:00,137,64.0,north
4,P00004,baseline,2026-02-25,10:55,145,106.2,north
...,...,...,...,...,...,...,...
57366,P24976,week12,2026-05-14,10:50,153,78.3,south
57367,P19276,baseline,2026-01-22,15:35,160,122.2,south
57368,P21334,week12,2026-05-05,09:40,129,98.2,south
57369,P03524,baseline,2026-02-06,10:05,160,117.6,south


### 2.3 Audit the raw tables

In [145]:
assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."
# Lists to map values
    # For gender
gender_codes = ['F', 'M', 'X']
    # For blanks
missing_codes = ['']
    # For particular visits
visit_nobaseline12 = ['baseline', 'week12']
    # For BP outliers
bp_outliers = ((visits_raw['sbp_mmhg'].astype(int) < 70) | (visits_raw['sbp_mmhg'].astype(int) > 250)) 

# Create a copy of the 'patients.csv' within `raw` json
raw_patients = raw_snapshot['patients'].copy()

# Create a cleaned version of 'site_code' variable within `raw_patients`
raw_patients['site_code_clean'] = raw_patients['site_code'].str.upper().str.replace(' ', '').str.replace('-', '')

# Count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and
    # count, and save it to AUDIT_PATH.
issue_audit = pd.DataFrame({
    'issue': ['patients: gender values that are not a code',
              'patients: empty gender',
              'patients: birth_year sentinel 9999', 
              'patients: site codes needing normalization',
              'sites: rows with a repeated site_code',
              'visits: visit labels that are not baseline or week12',
              'visits: sbp sentinel 999',
              'visits: sbp outside 70 through 250',
              'visits: date parse failures',
              'visits: rows in repeated patient_id and visit pairs'],
    'count': [len([(raw_patients['gender'].str.strip().notnull()) & (~raw_patients['gender'].str.strip().isin(gender_codes))]), 
    ((raw_patients['gender'].str.strip().isin(missing_codes)) | (raw_patients['gender'].str.strip().isna())).sum(),
    (raw_patients['birth_year'] == '9999').sum(),
    (raw_patients['site_code'] != raw_patients['site_code_clean']).sum(),
    len(raw['sites'].duplicated(subset=["site_code"], keep = False)),
    len(visits_raw[~visits_raw['visit'].isin(visit_nobaseline12)]),
    (visits_raw['sbp_mmhg'] == '999').sum(),
    len(visits_raw[bp_outliers]) - (visits_raw['sbp_mmhg'] == '999').sum(),
    (~visits_raw['visit_date'].astype(str).str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")).sum(),
    (visits_raw[visits_raw['visit'].isin(visit_nobaseline12)].duplicated(subset=['patient_id', 'visit'], keep = False)).sum()]
})

issue_audit.to_csv(AUDIT_PATH, index = False)

## Task 3: Clean

### 3.1 Clean the patients

In [146]:
# List the gender spellings with value_counts(), then build patients_clean from
patients_clean = raw["patients"].copy(deep=True) # with steps 1 to 5 in README Task 3.1.

    # a) Remove exact duplicate rows, keeping the first.
patients_clean = patients_clean.drop_duplicates(keep = "first")
    # b) gender: list the spellings with value_counts(), then strip and lowercase them and map each to its code from the data dictionary with a dictionary and Series.map() (Lecture 05), such as {"female": "F", "woman": "F", ...}. 
        # A spelling the dictionary leaves out, such as unknown or an empty value, becomes missing. Convert the result with astype("category").
patients_clean['gender'].value_counts()
        # Map every spelling of gender to F, M, or X with a dictionary
gender_codes = {
    'f': 'F',
    'female': 'F',
    'woman': 'F',
    'm': 'M',
    'male': 'M',
    'man': 'M',
    'x': 'X',
    'nb': 'X',
    'nonbinary': 'X'
}
        # Clean gender
patients_clean["gender"] = patients_clean["gender"].str.strip().str.lower().str.replace(' ', '').str.replace('-', '').map(gender_codes).astype("category")

    # c) `birth_year`: the 9999 sentinel becomes missing; dtype Int64
patients_clean['birth_year'] = patients_clean['birth_year'].astype('Int64').replace(9999.0, np.nan)

    # d) `site_code`: uppercase, then remove every hyphen and space with .str.replace("-", "") and .str.replace(" ", "")
patients_clean['site_code'] = patients_clean['site_code'].str.upper().str.replace(' ', '').str.replace('-', '')

    # e) Renumber the rows with reset_index(drop=True)
patients_clean.reset_index(drop = True)

    # f) Save the patients summary to output/patients_summary.csv with these metrics, in this order:
patients_summary = pd.DataFrame({
    'metric' : ['patients',
                'birth_year missing',
                'birth_year min',
                'birth_year max',
                'site_code values',
                'patients at N01',
                'patients at S02',
                'patients at E04'],
    'value' : [len(patients_clean),
               patients_clean["birth_year"].isna().sum(),
               patients_clean["birth_year"].min(),
               patients_clean["birth_year"].max(),
               patients_clean["site_code"].nunique(),
               (patients_clean["site_code"] == "N01").sum(),
               (patients_clean["site_code"] == "S02").sum(),
               (patients_clean["site_code"] == "E04").sum()]
})

patients_summary

,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


In [147]:
# a) Show patients_clean["gender"].cat.categories, 
patients_clean["gender"].cat.categories

# b) Save the patients summary to PATIENTS_SUMMARY_PATH, and 
patients_clean.to_csv(PATIENTS_SUMMARY_PATH, index = False)

# c) Save the gender value_counts(dropna=False) with a code column from .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.
gender_counts = patients_clean["gender"].value_counts(dropna = False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index = False)

### 3.2 Clean the visits

In [148]:
# Build visits_clean from
visits_clean = visits_raw.copy(deep = True) # with steps 1 to 5 in README Task 3.2.

    # a) Remove exact duplicate rows, keeping the first.
visits_clean.drop_duplicates(keep = "first")

    # b) visit: list the spellings with value_counts(), then strip and lowercase them and map each to baseline or week12 with a dictionary and Series.map(), as for gender.
visits_clean['visit'].str.strip().str.lower().value_counts()

        # Map every spelling of gender to F, M, or X with a dictionary
visit_codes = {
    'baseline': 'baseline',
    'bl': 'baseline',
    'week12': 'week12',
    'wk12': 'week12',
    'week 12': 'week12'
}
        # Clean visits
visits_clean['visit'] = visits_clean['visit'].str.strip().str.lower().map(visit_codes).astype("category")

    # c) sbp_mmhg: convert with pd.to_numeric(..., errors="coerce"); keep values from 70 through 250, and make everything else, 999 included, missing; dtype Int64
visits_clean['sbp_mmhg'] = pd.to_numeric(visits_clean['sbp_mmhg'], errors = "coerce")
visits_clean['sbp_mmhg'] = visits_clean['sbp_mmhg'].mask(~(visits_clean['sbp_mmhg'].between(70, 250)), np.nan).astype('Int64')

    # d) weight_kg: convert with pd.to_numeric(..., errors="coerce"); an empty value stays missing.
visits_clean['weight_kg'] = pd.to_numeric(visits_clean['weight_kg'], errors = "coerce")

    # e) visit_date: keep a value only when its text is exact YYYY-MM-DD and the date exists on the calendar; anything else becomes missing.
visits_clean['visit_date'] = visits_clean['visit_date'].mask(~(visits_clean['visit_date'].astype(str).str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")), np.nan)


In [ ]:
# Step 6, 
    # a) show the repeated patient_id and visit pairs, 
visits_clean.duplicated(subset=['patient_id', 'visit'], keep = False)

    # b) then keep the later reading.
visits_clean = visits_clean.sort_values("visit_time").drop_duplicates(subset=["patient_id", "visit"], keep="last")


In [154]:
# Step 7
    # a) sort, renumber, keep the six columns, 
visits_clean = visits_clean.sort_values(['patient_id', 'visit']).reset_index(drop = True)

    # b) Create a summary of the visits table, 
visits_summary = pd.DataFrame({
    'metric': ['visits', 
               'baseline visits',
               'week12 visits',
               'visit_date missing',
               'visit_date min',
               'visit_date max',
               'sbp_mmhg missing',
               'sbp_mmhg min',
               'sbp_mmhg max',
               'weight_kg missing',
               'weight_kg mean',
               'visits from visits_north.csv',
               'visits from visits_south.csv'],
    'value': [len(visits_clean),
              (visits_clean['visit'] == 'baseline').sum(),
              (visits_clean['visit'] == 'week12').sum(),
              visits_clean['visit_date'].isna().sum(),
              visits_clean['visit_date'].min(),
              visits_clean['visit_date'].max(),
              visits_clean['sbp_mmhg'].isna().sum(),
              visits_clean['sbp_mmhg'].min(),
              visits_clean['sbp_mmhg'].max(),
              visits_clean['weight_kg'].isna().sum(),
              round(visits_clean["weight_kg"].mean(), 2),
              (visits_clean['source_file'] == "north").sum(),
              (visits_clean['source_file'] == "south").sum()]
})

# then save the visits summary to VISITS_SUMMARY_PATH (README Task 3.2).
visits_summary.to_csv(VISITS_SUMMARY_PATH, index = False)

### 3.3 Record the decisions

In [157]:
# One row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
    # a) Create 4 columns
decision_table = pd.DataFrame(columns=["table", "issue", "decision", "reason"])
    # b) Which tables did these issues arise from
decision_table["table"] = ['patients', 'patients','patients','patients','visits','visits','visits', 'visits', 'site']
    # c) What were the issues faced
decision_table['issue'] = ['exact duplicate rows',
                           'gender spellings that are not a code',
                           'site codes needing normalization',
                           'birth_year sentinel 9999',
                           'weight column named wt_kg in visits_south.csv',
                           'sbp sentinel 999 and readings outside 70 through 250',
                           'impossible and loosely written visit dates',
                           'repeated patient_id and visit pairs',
                           'repeated site_code rows']
    # d) What decision was made
decision_table['decision'] = [
    'Remove exact duplicate rows, keeping the first.',
    'Encoded gender into specific values: "F", "M" or "X"',
    'Made encodings for site consistent',
    'Converted birth year encoded as "9999" as missing',
    'Renamed "wt_kg" as "weight_kg"',
    'Converted sbp outliers including "999" as missing and made sbp numerical',
    'Kept visit date values with a specific format of "YYYY-MM-DD", encoded impossible dates as missing',
    'Removed duplicates, kept the most recent visit record for each patient',
    'Cleaned site code values by removing hyphens and spaces'
]
    # e) Reason for decision being made
decision_table['reason'] = [
    'Repeated rows will skew downstream analysis, first entry of patient record likely to be most reliable on patient characteristics',
    'Easier for future summarization of gender composition of patient cohort',
    'Easier for future summarization of study sites',
    'As inclusion of impossible birth years will skew downstream analyses',
    'Will help merging visit tables collected north vs south locations',
    'as inclusion of impossible SBP values will skew downstream analyses',
    'As inclusion of visit dates will skew downstream analyses',
    'Most recent visit record more reflective of current patient status',
    'Will help summarization of study sites'
]   

decision_table.to_csv(DECISION_LOG_PATH, index = False)

## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [ ]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = None

### 4.2 Check the keys before joining

In [ ]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [ ]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [ ]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [ ]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [ ]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [ ]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [ ]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [ ]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.